# CS 336 playground

*  TODO look up "min p sampling"

In [1]:
# from the 7.2 secion

vocab_size = 10000
context_length = 256
d_model = 512
d_ff = 1344
rope_theta = 10000
num_layers = 4
num_heads = 16

!wget -q -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O /tmp/tiny_shakespeare.txt

if shakespeare:=False:
    vocab_source = "/tmp/tiny_shakespeare.txt"
    vocab_cache = f"/tmp/bpe_shakespeare_{vocab_size}.saved"
    training_txt = "tiny_shakespeare.txt"
    training_uint16 = "/tmp/tiny_shakespeare.uint16"

if tinystories:=True:
    vocab_source = '../data/TinyStoriesV2-GPT4-valid.txt'
    vocab_cache = f"/tmp/bpe_tinystories_{vocab_size}.saved"
    training_txt = '../data/TinyStoriesV2-GPT4-valid.txt'
    training_uint16 = "/tmp/tiny_stories_validation.uint16"



In [2]:
import torch
import os
from cs336_basics.ron_bpe_tokenizer import RonBPETokenizer
from cs336_basics.ron_train_bpe import train_bpe

## BPE training

In [3]:
from cs336_basics.ron_train_bpe import train_bpe
"""
    vocab size of 1000 on TinyStoriesV2-GPT4-valid.txt  
    isn't horrible for CPU-based interactive notebook use.
    It takes 16 seconds to train and makes words like " Sally" and " helped" 
    into single tokens. Maybe roughly the vocab of a preschooler :)

    vocab size of 10000 on TinyStoriesV2-GPT4-valid.txt takes like 
    three minutes. and makes words like " sinking" and " ribbit".

    Shakespeare's about 3 minutes as well. 
"""
if not os.path.exists(vocab_cache):
    vocab,merges = train_bpe(vocab_source,10000,[])
    obj = {"vocab": vocab,"merges": merges}
    torch.save(obj, vocab_cache)
else:
    obj = torch.load(vocab_cache)
    vocab,merges = obj['vocab'],obj['merges']
print(merges[-10:])

[(b' sc', b'ientist'), (b' sc', b'ience'), (b' satis', b'faction'), (b' sa', b'ves'), (b' s', b'uspicious'), (b' s', b'ly'), (b' s', b'inking'), (b' ro', b'les'), (b' ro', b'amed'), (b' ribb', b'it')]


## BPE class

In [4]:
from cs336_basics.ron_bpe_tokenizer import RonBPETokenizer
tokenizer = RonBPETokenizer(vocab,merges)
tokens    = list(tokenizer.encode_iterable(["Hello world"," ","Good bye"]))
print(tokens)
print(tokenizer.decode(tokens))

[1207, 1597, 32, 2098, 5452]
Hello world Good bye


### Make a nice training dataset with that tokenizer

In [5]:
import numpy as np

def tokenize_file_to_numpy(in_path, out_path, tokenizer, dtype=np.uint16):
    tokens = []
    os.remove(out_path)
    with open(out_path, "ab") as fout:
        with open(in_path, "r", encoding="utf-8") as fin:
            for line in fin:
                toks = tokenizer.encode(line)
                toks_np = np.asarray(toks, dtype=dtype)
                fout.write(toks_np.tobytes())

# about 7 seconds with a 1000 vocab
# about 30 seconds with a 10000 vocab

if regen_numpy_ds:=True:
    tokenize_file_to_numpy(training_txt,training_uint16,tokenizer)

In [6]:
token_ds = np.memmap(training_uint16, dtype=np.uint16, mode="r")
token_ds

memmap([117, 865, 498, ..., 384, 382,  46], shape=(5512360,), dtype=uint16)

## TRAIN!!!

In [7]:
import cs336_basics.ron_transformer_lm as ron_transformer_lm

# d_model = 512
# num_heads = 8
# d_ff = 128
# max_seq_len = 1000
# rope_theta = 10000
# vocab_size = 1000
# context_length = 1000
# num_layers = 4

tlm = ron_transformer_lm.TransformerLM(
    d_model=d_model,
    num_heads=num_heads,
    d_ff = d_ff,
    max_seq_len=context_length,
    theta=rope_theta,
    vocab_size=vocab_size,
    context_length=context_length,
    num_layers=num_layers
    )
tlm.to('cuda')
tlm.forward([tokenizer.encode("Hello World")])

tensor([[[ 0.0803,  0.0537, -0.3627,  ...,  0.6282,  0.3667, -0.0311],
         [-0.0629,  0.2205, -0.2567,  ...,  0.1993, -0.0574,  0.1106],
         [ 0.0822,  0.2164, -0.3196,  ...,  0.8223,  0.2949,  0.0289],
         [ 0.0178,  0.2694,  0.0509,  ...,  0.5086,  0.1974,  0.4901]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

## My AdamW

In [8]:
import cs336_basics.ron_adamw_optimizer as ron_adamw_optimizer
my_adamw = ron_adamw_optimizer.AdamW(tlm.parameters())

## My Data Loader

In [9]:
from cs336_basics.ron_data_loader import get_batch

def get_training_batch():
    return get_batch(token_ds,10,context_length,'cpu')
get_training_batch()


(tensor([[ 350,  434,  382,  ...,  632, 5901,  329],
         [ 262,  958,   46,  ...,  686, 8175,  372],
         [ 375,  373,   10,  ...,  326,  951,  365],
         ...,
         [ 689, 1905,  938,  ...,  580,  262, 2401],
         [ 324,  709,  329,  ...,   44,  262,  636],
         [ 258, 1984,   46,  ...,   10,  650,  580]], dtype=torch.uint16),
 tensor([[ 434,  382,  405,  ..., 5901,  329, 1186],
         [ 958,   46,  317,  ..., 8175,  372,  262],
         [ 373,   10,   10,  ...,  951,  365,  265],
         ...,
         [1905,  938,   46,  ...,  262, 2401,  282],
         [ 709,  329,  340,  ...,  262,  636, 4572],
         [1984,   46,  435,  ...,  650,  580,  372]], dtype=torch.uint16))

## Training Loop

In [10]:
import torch
import torch.nn.functional as F
import time
def train(model, optimizer, get_batch, device="cuda", 
          num_iters=2000):
    model.to(device)
    model.train()
    t0 = time.time()
    for it in range(num_iters):
        x, y = get_batch()
        x = torch.tensor(x, dtype=torch.long, device=device)
        y = torch.tensor(y, dtype=torch.long, device=device)

        logits = model(x)            # shape: (batch, seq, vocab)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if it % 100 == 0:
            print(f"iter {it} | loss {loss.item():.4f} in {time.time() - t0} secs")

train(tlm,my_adamw,get_training_batch)


/tmp/ipykernel_2549152/2104460885.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.long, device=device)
/tmp/ipykernel_2549152/2104460885.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y, dtype=torch.long, device=device)


iter 0 | loss 9.2548 in 0.39628028869628906 secs
iter 100 | loss 3.6551 in 11.54191255569458 secs
iter 200 | loss 3.1199 in 22.724795818328857 secs
iter 300 | loss 3.0057 in 34.175777196884155 secs
iter 400 | loss 2.8591 in 45.37292432785034 secs
iter 500 | loss 2.5867 in 56.54169273376465 secs
iter 600 | loss 2.5539 in 67.77764797210693 secs
iter 700 | loss 2.6610 in 79.13795757293701 secs
iter 800 | loss 2.7645 in 90.38015151023865 secs
iter 900 | loss 2.6153 in 101.73332977294922 secs
iter 1000 | loss 2.3523 in 113.28367328643799 secs
iter 1100 | loss 2.6470 in 124.6394624710083 secs
iter 1200 | loss 2.6876 in 135.9823718070984 secs
iter 1300 | loss 2.4609 in 147.37154746055603 secs
iter 1400 | loss 2.3561 in 159.30867290496826 secs
iter 1500 | loss 2.1876 in 170.82082343101501 secs
iter 1600 | loss 2.2799 in 182.2522132396698 secs
iter 1700 | loss 2.3639 in 193.50713896751404 secs
iter 1800 | loss 2.1550 in 204.7859070301056 secs
iter 1900 | loss 2.1619 in 215.9887363910675 secs


In [15]:
import torch
predictions = tlm.forward([tokenizer.encode("Hello World")])
predictions

tensor([[[-5.0237, -5.3700, -5.6582,  ..., -5.3895, -4.8646, -4.4940],
         [-1.9553, -2.3798, -2.3391,  ..., -1.5179, -1.9361, -1.7673],
         [-3.4015, -3.6887, -3.8551,  ..., -3.1735, -3.1456, -3.0646],
         [-4.3214, -4.6662, -4.6535,  ..., -3.6596, -3.7660, -3.9862]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [16]:
last_logits = predictions[0, -1]
next_token_id = last_logits.argmax().item()
tokenizer.decode([next_token_id])

'.'

In [17]:
num_tokens_to_generate = 50

# Convert to a list for easy appending
generated_tokens = tokenizer.encode("The boy and")

for _ in range(num_tokens_to_generate):
    seq_len = len(generated_tokens)
    tokens_tensor = torch.tensor([generated_tokens])

    with torch.no_grad():
        predictions = tlm.forward(tokens_tensor)  # (1, seq_len, vocab_size)

    last_logits = predictions[0, -1]
    next_token_id = last_logits.argmax().item()
    generated_tokens.append(next_token_id)

# Decode the whole sequence
decoded_text = tokenizer.decode(generated_tokens)
print(decoded_text)


The boy and the boy were very happy. They thanked the boy and the boy. They all learned that sharing is good.
<|endoftext|>
Once upon a time, there was a little girl named Lily. She loved to play with her toys and eat yummy


In [18]:
num_tokens_to_generate = 50

# Convert to a list for easy appending
generated_tokens = tokenizer.encode("The boy and")

for _ in range(num_tokens_to_generate):
    seq_len = len(generated_tokens)
    tokens_tensor = torch.tensor([generated_tokens])

    with torch.no_grad():
        predictions = tlm.forward(tokens_tensor)  # (1, seq_len, vocab_size)

    last_logits = predictions[0, -1]
    #next_token_id = last_logits.argmax().item()


    probs = F.softmax(last_logits, dim=-1)
    next_token_id = torch.multinomial(probs, num_samples=1).item()
    generated_tokens.append(next_token_id)

# Decode the whole sequence
decoded_text = tokenizer.decode(generated_tokens)
print(decoded_text)


The boy and the old lady were very happy. They picked a doll blueberrytern and showed it to her friends. They showed it to her mommy. Mom smiled, said, "Yes, this is a good rag. Can you keep the doll, please?" Daddy
